# 🚇 Analyse des Flux de Mobilité

**Objectif** : croiser les données de mobilité avec les indicateurs démographiques pour identifier les zones sous-équipées.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# Données communes simulées (en production : charger depuis INSEE + IDFM)
communes_data = {
    'commune': ['Noisy-le-Grand','Montreuil','Vincennes','Saint-Denis',
                 'Nanterre','Créteil','Boulogne-Billancourt','Versailles',
                 'Argenteuil','Évry','Massy','Cergy'],
    'population': [73000, 109000, 49000, 115000, 96000, 93000, 121000, 86000,
                   116000, 60000, 48000, 65000],
    'validations_tc_annuelles': [2.1e6, 4.5e6, 3.8e6, 6.2e6, 3.1e6, 2.9e6,
                                  5.4e6, 3.7e6, 2.3e6, 1.8e6, 2.6e6, 1.5e6],
    'km_pistes_cyclables': [18, 42, 28, 65, 31, 24, 78, 45, 15, 22, 35, 12],
    'revenu_median': [23000, 21000, 38000, 18000, 28000, 20000, 45000, 36000,
                       17000, 19000, 31000, 22000],
    'taux_chomage': [14.2, 16.8, 6.1, 19.5, 11.3, 17.2, 5.8, 7.4,
                      20.1, 18.9, 9.2, 15.6]
}
df = pd.DataFrame(communes_data)

# Indicateurs dérivés
df['validations_par_habitant'] = df['validations_tc_annuelles'] / df['population']
df['km_velo_par_1000hab'] = df['km_pistes_cyclables'] / (df['population'] / 1000)

print(df[['commune','population','validations_par_habitant','km_velo_par_1000hab']].to_string(index=False))

## 1. Matrice de corrélation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Corrélation
num_cols = ['population','validations_par_habitant','km_velo_par_1000hab',
            'revenu_median','taux_chomage']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Matrice de corrélation', fontsize=13, fontweight='bold')

# Scatter : revenu vs validations TC
axes[1].scatter(df['revenu_median'], df['validations_par_habitant'],
                s=df['population']/1000, alpha=0.7,
                c=df['taux_chomage'], cmap='RdYlGn_r')
for _, row in df.iterrows():
    axes[1].annotate(row['commune'], (row['revenu_median'], row['validations_par_habitant']),
                     fontsize=8, ha='left')
axes[1].set_xlabel('Revenu médian (€)')
axes[1].set_ylabel('Validations TC / habitant')
axes[1].set_title('Revenu vs Usage des Transports en Commun\n(taille = population, couleur = chômage)',
                   fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/03_correlations_mobilite.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Segmentation des communes (K-Means)

In [ ]:
features = ['validations_par_habitant', 'km_velo_par_1000hab',
            'revenu_median', 'taux_chomage']

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

# Méthode du coude
inerties = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_ for k in range(2, 8)]

plt.figure(figsize=(8, 4))
plt.plot(range(2, 8), inerties, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k optimal = 3')
plt.xlabel('Nombre de clusters')
plt.ylabel('Inertie')
plt.title('Méthode du coude — Choix du nombre de clusters', fontweight='bold')
plt.legend()
plt.savefig('outputs/04_methode_coude.png', dpi=150, bbox_inches='tight')
plt.show()

# Clustering final
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X)

LABELS = {0: '🟡 Mobilité moyenne', 1: '🔴 Sous-équipée & précaire', 2: '🟢 Bien desservie & aisée'}
df['profil'] = df['cluster'].map(LABELS)

print('\n=== PROFILS TERRITORIAUX ===')
for label in LABELS.values():
    communes = df[df['profil'] == label]['commune'].tolist()
    print(f'\n{label} : {", ".join(communes)}')

## 3. Recommandations décisionnelles

| Profil | Communes | Action prioritaire |
|--------|----------|-------------------|
| 🔴 Sous-équipée & précaire | Saint-Denis, Argenteuil, Évry | Renforcement offre TC + pistes cyclables |
| 🟡 Mobilité moyenne | Noisy-le-Grand, Créteil, Cergy | Intermodalité TC-vélo à développer |
| 🟢 Bien desservie | Boulogne, Vincennes, Versailles | Optimisation flux existants |

➡️ Suite : `03_cartographie_interactive.ipynb` — visualisation SIG des résultats